In [1]:
import pandas as pd
from google.colab import drive

# ==============================================================================
# PHASE 2: ALGORITHMIC YIELD & MOMENTUM ANALYSIS
# ==============================================================================


drive.mount('/content/drive')

print("Ingesting normalized time-series data...")
input_path = '/content/drive/MyDrive/project_portfolio/housing/zillow_arbitrage_clean.csv'
df = pd.read_csv(input_path)
df['Date'] = pd.to_datetime(df['Date'])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ingesting normalized time-series data...


In [4]:
# ------------------------------------------------------------------------------
# FEATURE ENGINEERING: QUANTIFYING THE ARBITRAGE
# ------------------------------------------------------------------------------
print("[ANALYSIS] Quantifying Market Yields (Price-to-Rent Ratios)...")

# Institutional investors evaluate property on an annual yield basis.
df['Annual_Rent'] = df['Rent_Value'] * 12

# The Price-to-Rent Ratio is our primary 'Arbitrage' metric.
# A lower ratio indicates a higher cap rate (potential ROI).
df['Price_to_Rent_Ratio'] = df['Home_Value'] / df['Annual_Rent']

# ------------------------------------------------------------------------------
# TIME-SERIES ANALYSIS: YoY MOMENTUM
# ------------------------------------------------------------------------------
print("[ANALYSIS] Calculating YoY Momentum (Seasonality-Adjusted Growth)...")

# We sort by market and date to ensure the rolling 12-month window is accurate.
df = df.sort_values(by=['RegionName', 'Date'])

# We use a 12-period shift to calculate Year-over-Year (YoY) variance.
# This neutralizes the seasonality inherent in the winter/summer real estate cycles.
df['Home_YoY_Growth'] = df.groupby('RegionName')['Home_Value'].pct_change(periods=12)
df['Rent_YoY_Growth'] = df.groupby('RegionName')['Annual_Rent'].pct_change(periods=12)

# ------------------------------------------------------------------------------
# SNAPSHOT GENERATION: PERFORMANCE REPORTING
# ------------------------------------------------------------------------------
print("[ANALYSIS] Isolating latest market snapshot for BI deployment...")

latest_date = df['Date'].max()
df_current = df[df['Date'] == latest_date].copy()

# Exporting the master file (for timeline charts) and a snapshot (for the Buy List)
master_export = '/content/drive/MyDrive/project_portfolio/housing/zillow_final_clean_data.csv'
df.to_csv(master_export, index=False)

print("-" * 50)
print(f"MARKET SNAPSHOT: {latest_date.strftime('%B %Y')}")
print(f"Total Metropolitan Areas Tracked: {len(df_current)}")
print("-" * 50)

# Quick validation: Markets with the highest Rent Momentum
print("\nHIGH MOMENTUM RENTAL MARKETS (Top 5 YoY Growth):")
performance_check = df_current.sort_values(by='Rent_YoY_Growth', ascending=False).head()
print(performance_check[['RegionName', 'Price_to_Rent_Ratio', 'Rent_YoY_Growth']])

[ANALYSIS] Quantifying Market Yields (Price-to-Rent Ratios)...
[ANALYSIS] Calculating YoY Momentum (Seasonality-Adjusted Growth)...
[ANALYSIS] Isolating latest market snapshot for BI deployment...
--------------------------------------------------
MARKET SNAPSHOT: January 2026
Total Metropolitan Areas Tracked: 704
--------------------------------------------------

HIGH MOMENTUM RENTAL MARKETS (Top 5 YoY Growth):
               RegionName  Price_to_Rent_Ratio  Rent_YoY_Growth
47384         Abilene, TX             9.366639         0.305708
47356          Monroe, LA            11.907372         0.258384
47622  Mount Pleasant, MI            15.369408         0.213941
47667         Ontario, OR            20.469059         0.151966
47647         Rutland, VT            16.195764         0.144862
